In [0]:
from datetime import date
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
%run ../../utils/writer_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()
earliest_date = date.fromisoformat('2020-01-01')

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', 'False').lower() == "true"
START_DATE = earliest_date
END_DATE = today

CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
SOURCE_TABLE_CONF = {
    "table": "pyspark_dim_date",
    "schema": "gold",
}

TARGET_TABLE_CONF = {
    "table": "pyspark_dim_date",
    "schema": "gold",
    "merge_keys": ["date_key"],
    "primary_key": "date_key"
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
if not INITIAL_RUN:
    START_DATE = (spark
                        .table(f"""{SOURCE_TABLE_CONF.get("schema")}
                                .{SOURCE_TABLE_CONF.get("table")}""")
                        .agg(F.max(F.col("full_date"))).first()[0])

In [0]:
print(START_DATE)

In [0]:
date_range_df = spark.sql(f"""
    SELECT explode(sequence(DATE '{START_DATE}', DATE '{END_DATE}')) AS full_date
""")

In [0]:
date_df = (date_range_df
                .withColumn("date_key", 
                       F.date_format(F.col("full_date"), "yyyyMMdd").cast(IntegerType()))
                .select(F.col("date_key"), 
                    F.col("full_date"))
                .withColumn("year", F.year(F.col("full_date")))
                .withColumn("month",      F.month("full_date"))
                .withColumn("month_name", F.date_format("full_date", "MMMM"))
                .withColumn("iso_week",   F.weekofyear("full_date"))
                .withColumn("day_of_week", F.dayofweek("full_date"))
                .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7)))   

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"
    primary_key = TARGET_TABLE_CONF.get('primary_key')

    (date_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY AUTO")
    spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {primary_key} SET NOT NULL")
    spark.sql(f"ALTER TABLE {target_table} ADD CONSTRAINT {primary_key}_pk PRIMARY KEY ({primary_key})")
    
    dbutils.notebook.exit(f"Initial Table: {target_table} Setup Finished Successfully")

In [0]:
upsert_to_delta(date_df, TARGET_TABLE_CONF)